In [1]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import ast
from collections import defaultdict

In [2]:
RRS_filtered = pd.read_csv('Req_Res_Con_Tables/RRS_filtered.csv', index_col = False)
for col in ['Requirements', 'Responsibilities', 'Conditions']:
    RRS_filtered[col] = RRS_filtered[col].apply(ast.literal_eval)

requirements = pd.read_csv('Req_Res_Con_Tables/Requirements_filtered.csv', index_col = False)['Требование']
responsibilities = pd.read_csv('Req_Res_Con_Tables/Responsibilities_filtered.csv', index_col = False)['Обязанность']
conditions = pd.read_csv('Req_Res_Con_Tables/Conditions_filtered.csv', index_col = False)['Условие']

### Threshold = 0.3

In [3]:
with open("Relevance Data/relevant_requirements_0_30.pkl", "rb") as f:
    relevant_requirements_0_30 = pickle.load(f)

with open("Relevance Data/relevant_responsibilities_0_30.pkl", "rb") as f:
    relevant_responsibilities_0_30 = pickle.load(f)

with open("Relevance Data/relevant_conditions_0_30.pkl", "rb") as f:
    relevant_conditions_0_30 = pickle.load(f)

### Threshold = 0.25

In [3]:
with open("Relevance Data/relevant_requirements_0_25.pkl", "rb") as f:
    relevant_requirements_0_25 = pickle.load(f)

with open("Relevance Data/relevant_responsibilities_0_25.pkl", "rb") as f:
    relevant_responsibilities_0_25 = pickle.load(f)

with open("Relevance Data/relevant_conditions_0_25.pkl", "rb") as f:
    relevant_conditions_0_25 = pickle.load(f)

# 1) With weights

### Threshold = 0.30

### Расчёт весов вакансий

In [ ]:
vacancy_weights = dict()
for id in RRS_filtered.Id:
    weight = (len(relevant_requirements_0_30[id]) +
                      len(relevant_responsibilities_0_30[id]) +
                      len(relevant_conditions_0_30[id])) / 3
    vacancy_weights[id] = weight

### Расчёт матрицы релевантности

In [37]:
weighted_corelevance_matrix_0_30 = np.zeros((150, 150, 150), dtype = float)
normalization_coefficient = 0
for vac_id, vac_weight in tqdm(vacancy_weights.items()):
    req_mask = np.array([
        req in relevant_requirements_0_30[vac_id]
        for _, req in requirements.items()
    ], dtype=float)

    res_mask = np.array([
        res in relevant_responsibilities_0_30[vac_id]
        for _, res in responsibilities.items()
    ], dtype=float)

    con_mask = np.array([
        con in relevant_conditions_0_30[vac_id]
        for _, con in conditions.items()
    ], dtype=float)

    temp = np.einsum('i,j,k->ijk', req_mask, res_mask, con_mask)

    weighted_corelevance_matrix_0_30 += temp / vac_weight
    normalization_coefficient += (1 / vac_weight)

weighted_corelevance_matrix_0_30 = weighted_corelevance_matrix_0_30 / normalization_coefficient
np.save('weighted_corelevance_matrix_0_30.npy', weighted_corelevance_matrix_0_30)

100%|██████████| 6914/6914 [19:01<00:00,  6.06it/s]


# 2) No weights

### Threshold = 0.30

### Расчёт матрицы релевантности

In [38]:
no_weighting_corelevance_matrix_0_30 = np.zeros((150, 150, 150), dtype = float)
for vac_id in tqdm(RRS_filtered.Id):
    req_mask = np.array([
        req in relevant_requirements_0_30[vac_id]
        for _, req in requirements.items()
    ], dtype=float)

    res_mask = np.array([
        res in relevant_responsibilities_0_30[vac_id]
        for _, res in responsibilities.items()
    ], dtype=float)

    con_mask = np.array([
        con in relevant_conditions_0_30[vac_id]
        for _, con in conditions.items()
    ], dtype=float)

    temp = np.einsum('i,j,k->ijk', req_mask, res_mask, con_mask)

    no_weighting_corelevance_matrix_0_30 += temp

no_weighting_corelevance_matrix_0_30 = no_weighting_corelevance_matrix_0_30 / len(RRS_filtered.Id)
np.save('no_weighting_corelevance_matrix_0_30.npy', no_weighting_corelevance_matrix_0_30)

100%|██████████| 6914/6914 [15:18<00:00,  7.52it/s]


# 3) Kulchinsky strategy

In [4]:
def build_inverted_index(dictionary):
    index = defaultdict(set)

    for vac_id, token_list in dictionary.items():
        for token in token_list:
            index[token].add(vac_id)

    return index

In [5]:
req_to_vac = build_inverted_index(relevant_requirements_0_30)
res_to_vac = build_inverted_index(relevant_responsibilities_0_30)
con_to_vac = build_inverted_index(relevant_conditions_0_30)

### Threshold = 0.30

In [8]:
kulch_corelevance_matrix_0_30 = np.zeros(
    (requirements.size, responsibilities.size, conditions.size), 
    dtype=float
)

req_items = list(requirements.items())
res_items = list(responsibilities.items())
con_items = list(conditions.items())

for req_idx, requirement in tqdm(req_items):
    A_set = req_to_vac.get(requirement, set())
    A = len(A_set)

    for res_idx, responsibility in res_items:
        B_set = res_to_vac.get(responsibility, set())
        B = len(B_set)

        for con_idx, condition in con_items:
            C_set = con_to_vac.get(condition, set())
            C = len(C_set)

            if A == 0 or B == 0 or C == 0:
                kulch_coefficient = 0
            else:
                ABC = len(A_set & B_set & C_set)
                kulch_coefficient = ABC * ((1 / A) + (1 / B) + (1 / C)) / 3

            kulch_corelevance_matrix_0_30[req_idx, res_idx, con_idx] = kulch_coefficient

np.save('kulch_corelevance_matrix_0_30.npy', kulch_corelevance_matrix_0_30)

  0%|          | 0/157 [00:00<?, ?it/s]

100%|██████████| 157/157 [00:11<00:00, 14.26it/s]
